In [0]:
TABLE_ACTIVITY_BRONZE = "customer_360.bronze.activities"
TABLE_SILVER_ACTIVITY = "customer_360.silver.activities"
TABLE_QUARANTINE_ACTIVITY = "customer_360.quarantine.activities"

TABLE_ACTIVITY_VIEWS = "customer_360.bronze.activity_views"

ACTIVITY_METRICS_TABLE = "customer_360.raw.activity_silver_metrics"

PATH_ACTIVITY_CHECKPOINTLOCATION_SILVER = (
    "/Volumes/customer_360/raw/source_files/checkpoints/silver/activities"
)

TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS customer_360.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS customer_360.quarantine
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER_ACTIVITY} (
    activity_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    activity_type STRING,
    activity_channel STRING,
    activity_time TIMESTAMP,
    description STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_QUARANTINE_ACTIVITY} (
    activity_id STRING,
    customer_id STRING,
    activity_type STRING,
    activity_channel STRING,
    activity_time TIMESTAMP,
    description STRING,
    updated_at TIMESTAMP,
    failure_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_ACTIVITY_VIEWS} (
    activity_id STRING NOT NULL,
    updated_at TIMESTAMP,
    viewed_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ACTIVITY_METRICS_TABLE} (
    metric_time TIMESTAMP,
    batch_id BIGINT,
    query_name STRING,
    total_records BIGINT,
    valid_records BIGINT,
    invalid_records BIGINT,
    duplicate_records BIGINT
)
USING DELTA
""")

In [0]:
activity_df = (
    spark
    .readStream
    .format("delta")
    .table(TABLE_ACTIVITY_BRONZE)
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime


def process_activity_dataframe(batch_df, batch_id):


    # Standardization + Data Quality Checks
    df = (
        batch_df

        .withColumn(
            "activity_id",
            trim(col("activity_id"))
        )

        .withColumn(
            "customer_id",
            trim(col("customer_id"))
        )

        .withColumn(
            "activity_type",
            trim(col("activity_type"))
        )

        .withColumn(
            "activity_channel",
            trim(col("activity_channel"))
        )

        .withColumn(
            "failure_reason",

            when(
                col("activity_id").isNull(),
                "activity_id is null"
            )

            .when(
                col("customer_id").isNull(),
                "customer_id is null"
            )

            .when(
                col("activity_type").isNull(),
                "activity_type is null"
            )

            .when(
                ~col("activity_type").isin(
                    "LOGIN",
                    "LOGOUT",
                    "PAGE_VIEW",
                    "PRODUCT_VIEW",
                    "SEARCH",
                    "ADD_TO_CART",
                    "REMOVE_FROM_CART",
                    "PURCHASE",
                    "EMAIL_OPEN",
                    "SUPPORT_REQUEST"
                ),
                "invalid activity_type"
            )

            .when(
                col("activity_channel").isNull(),
                "activity_channel is null"
            )

            .when(
                ~col("activity_channel").isin(
                    "WEB",
                    "MOBILE",
                    "APP",
                    "EMAIL",
                    "PHONE"
                ),
                "invalid activity_channel"
            )

            .when(
                col("activity_time").isNull(),
                "activity_time is null"
            )

            .when(
                col("updated_at").isNull(),
                "updated_at is null"
            )

            .otherwise(None)
        )
    )


   
    # Valid Data
    valid_data = (
        df
        .filter(
            col("failure_reason").isNull()
        )
        .drop("failure_reason")
    )


   
    # Invalid Data → Quarantine

    invalid_data = (
        df
        .filter(
            col("failure_reason").isNotNull()
        )
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    invalid_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(
            TABLE_QUARANTINE_ACTIVITY
        )


   
    # Within-Batch Duplicate Detection
    #
    # Duplicate Key:
    # activity_id + updated_at
   
    window = (
        Window
        .partitionBy(
            [
                "activity_id",
                "updated_at"
            ]
        )
        .orderBy(
            col("updated_at").asc()
        )
    )

    valid_data = (
        valid_data
        .withColumn(
            "rn",
            row_number().over(window)
        )
    )


    unique_data = (
        valid_data
        .filter(
            col("rn") == 1
        )
        .drop("rn")
    )


    duplicate_data = (
        valid_data
        .filter(
            col("rn") > 1
        )
        .drop("rn")
    )


   
    # Load Previously Processed Activities
   
    first_occurance = (
        spark
        .read
        .format("delta")
        .table(TABLE_ACTIVITY_VIEWS)
    )


   
    # Previously Seen Activities
   

    seen_data = (
        first_occurance
        .join(
            unique_data,
            on=[
                "activity_id",
                "updated_at"
            ],
            how="inner"
        )
        .select([
            "activity_id",
            "customer_id",
            "activity_type",
            "activity_channel",
            "activity_time",
            "description",
            "updated_at"
        ])
    )


   
    # New Activities
   

    silver_df = (
        unique_data
        .join(
            first_occurance,
            on=[
                "activity_id",
                "updated_at"
            ],
            how="left_anti"
        )
        .select([
            "activity_id",
            "customer_id",
            "activity_type",
            "activity_channel",
            "activity_time",
            "description",
            "updated_at"
        ])
    )


   
    # Write New Activities → Silver
   

    silver_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(
            TABLE_SILVER_ACTIVITY
        )

    valid_data_count = silver_df.count()


   
    # Duplicate Data → Quarantine
   

    quarantine_data = (
        duplicate_data
        .unionByName(seen_data)
        .withColumn(
            "failure_reason",
            lit("duplicate record")
        )
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(
            TABLE_QUARANTINE_ACTIVITY
        )

    duplicate_records = quarantine_data.count()


   
    # Store Processed Activities
   

    viewed_data = (
        silver_df
        .withColumn(
            "viewed_at",
            current_timestamp()
        )
        .select([
            "activity_id",
            "updated_at",
            "viewed_at"
        ])
    )

    viewed_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(
            TABLE_ACTIVITY_VIEWS
        )


   
    # Silver Batch Metrics
   

    total_records = batch_df.count()

    metric = [
        Row(
            metric_time=datetime.now(),
            batch_id=batch_id,
            query_name="activity_silver",
            total_records=total_records,
            valid_records=valid_data_count,
            invalid_records=(
                total_records - valid_data_count
            ),
            duplicate_records=duplicate_records
        )
    ]

    metric_df = spark.createDataFrame(metric)

    metric_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            ACTIVITY_METRICS_TABLE
        )

In [0]:
query = (
    activity_df
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(
        process_activity_dataframe
    )
    .option(
        "checkpointLocation",
        PATH_ACTIVITY_CHECKPOINTLOCATION_SILVER
    )
    .start()
)

query.awaitTermination()

In [0]:
import json
from pyspark.sql import Row
from datetime import datetime


metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="activity_silver",
            batch_id=int(
                progress["batchId"]
            ),
            input_rows=int(
                source.get(
                    "numInputRows",
                    0
                )
            ),
            input_rows_per_second=float(
                source.get(
                    "inputRowsPerSecond",
                    0.0
                )
            ),
            processed_rows_per_second=float(
                source.get(
                    "processedRowsPerSecond",
                    0.0
                )
            ),
            processing_time_ms=int(
                progress
                .get("durationMs", {})
                .get(
                    "triggerExecution",
                    0
                )
            )
        )
    )


if metrics:

    metrics_df = spark.createDataFrame(
        metrics
    )

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            TABLE_METRIC
        )

In [0]:
display(
    spark.sql(
        "SELECT * FROM customer_360.bronze.activities"
    )
)

display(
    spark.sql(
        "SELECT * FROM customer_360.silver.activities"
    )
)

display(
    spark.sql(
        "SELECT * FROM customer_360.bronze.activity_views"
    )
)

display(
    spark.sql(
        "SELECT * FROM customer_360.quarantine.activities"
    )
)

display(
    spark.sql(
        "SELECT * FROM customer_360.raw.activity_silver_metrics"
    )
)

display(
    spark.sql(
        "SELECT * FROM customer_360.raw.stream_metrics"
    )
)